# This notebooks investigates the claim that the $||u||_0^{\text{max}}$ term does nothing

In [ ]:
import numpy
import jax.numpy as jnp
import numpy as np
import seaborn as sns
import pandas as pd
from tqdm.autonotebook import tqdm
import matplotlib.pyplot as plt
import copy

from jaxopt import ScipyBoundedMinimize, LBFGS

rng = numpy.random.default_rng(0)

In [ ]:
u_dimension = 100
lam_1 = 1
max_l0_norm = 30
should_log = True


In [ ]:
def f1(v, rng, lam_1=1, max_l0_norm = 30,):
    u_to_s_function = lambda x: x

    u = rng.uniform(size=(u_dimension,)) * .1

    def objective(u):
        s = u_to_s_function(u)
        s_norm = jnp.linalg.norm(s)
        loss = 0
        loss += lam_1 * (max_l0_norm - jnp.sum(jnp.abs(u)))
        loss += jnp.dot(s, v) / (s_norm + 1e-10)
        return -loss.reshape()

    lb = jnp.zeros_like(u)
    ub = jnp.ones_like(u)

    bounds = (lb, ub)
    intermediate_xs = []
    runner = ScipyBoundedMinimize(fun=objective, method='l-bfgs-b', callback=lambda xk: intermediate_xs.append(xk) if should_log else None)
    result = runner.run(u, bounds=bounds)
    u = numpy.array(result.params)

    if u.max() > 0:
        u = numpy.array(u / u.max())


    u_orig = copy.deepcopy(u)
    idx = numpy.argsort(u)
    u[idx[:-max_l0_norm]] = 0

    u = u/u.max()
    return u, u_orig


def f2(v, rng, lam_1=1, max_l0_norm = 30):
    u_to_s_function = lambda x: x

    u = rng.uniform(size=(u_dimension,)) * .1

    def objective(u):
        s = u_to_s_function(u)
        s_norm = jnp.linalg.norm(s)
        loss = 0
        loss += lam_1 * (- jnp.sum(jnp.abs(u)))
        loss += jnp.dot(s, v) / (s_norm + 1e-10)
        return -loss.reshape()

    lb = jnp.zeros_like(u)
    ub = jnp.ones_like(u)

    bounds = (lb, ub)
    intermediate_xs = []
    runner = ScipyBoundedMinimize(fun=objective, method='l-bfgs-b', callback=lambda xk: intermediate_xs.append(xk) if should_log else None)
    result = runner.run(u, bounds=bounds)
    u = numpy.array(result.params)

    if u.max() > 0:
        u = numpy.array(u / u.max())


    u_orig = copy.deepcopy(u)
    idx = numpy.argsort(u)
    u[idx[:-max_l0_norm]] = 0

    u = u/u.max()
    return u, u_orig


In [ ]:
df = []

for _ in tqdm(range(10)):
    # v = numpy.zeros((100,1))
    # v[:30] = 1

    # v = rng.uniform(size=(u_dimension,))

    v = np.abs(rng.normal(size=(u_dimension,)))
    v[rng.permutation(u_dimension)[30:]] = 0

    v = v / np.linalg.norm(v)

    result = {}

    for max_l0_norm in (1000, 100, 10, 1, 0):
        u, u_orig = f1(v, copy.deepcopy(rng), lam_1=100, max_l0_norm=max_l0_norm)
        result[f'$||u||_0^{{ \\text{{max}} }} = {max_l0_norm}$'] = u

    u, u_orig = f2(v, copy.deepcopy(rng), lam_1=100, max_l0_norm=max_l0_norm)
    result['no $||u||_0^{\\text{max}}$'] = u

    df.append(result)
    rng.uniform()


result_columns = list(df[0].keys())
if 'v' in result_columns:
    result_columns.remove('v')

df = pd.DataFrame(df)


In [ ]:
metric_df = []

for c in result_columns:
    def f(x):
        ux = x/np.linalg.norm(x)
        uv = v/np.linalg.norm(v)
        return np.arccos(np.clip(np.dot(ux, uv), -1.0, 1.0)) * 180 / np.pi
    metric_df.append(df[c].apply(f))

metric_df = pd.DataFrame(metric_df).T
metric_df = metric_df[result_columns]



In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(metric_df, ax=ax)

for i, row in metric_df.iterrows():
    s = []
    ax.plot(range(len(row)), row, color='k', alpha=0.3)

ax.set_ylabel('angle from v (degrees)')
print(metric_df.mean())